# 0.0. Setup

In [47]:
import os
import gc
from pathlib import Path

import re
import shutil
import pickle
import importlib
import numpy as np
import pandas as pd
import openpyxl as opxl
import matplotlib.pyplot as plt

from difflib import SequenceMatcher
from sklearn.impute import SimpleImputer, KNNImputer

# None means 'no limit'
pd.set_option('display.max_columns', None)

PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / 'data'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
PUBLIC_DIR = PROJECT_DIR / 'data' / 'public'
PRIVATE_DIR = PROJECT_DIR / 'data' / 'private'
OUTPUT_DIR = PROJECT_DIR / 'output'

# 1.0. Load data
The .CSV file that will be loaded in this section was directly extracted by Claude Code from a .PDF file entitled "GASTPE Yearend Report SY 24-25 (Certfication Annexes).pdf" from DepEd GASS.

In [4]:
fname = "gastpe_certification_annexes_sy2425.csv"
fpath = str(PRIVATE_DIR / fname)
cert = pd.read_csv(fpath)
print(cert.shape)

(899, 5)


In [5]:
display(cert.head(3))

,annex,esc_school_id,region,school_name,rating
0,D,104519,Region I,Academia De Sta. Cecilia Foundation Inc.,3 Certified
1,D,104558,Region I,Bugallon Kidsworld Academy Inc.,2 Partial
2,D,104552,Region I,Ladder of Success Montessori School of Mangata...,2 Partial


## Inspect

In [10]:
cert_annexs = cert['annex'].unique()
display(cert_annexs)

cert_ratings = cert['rating'].unique()
display(cert_ratings)

cert_regions = cert['region'].unique()
display(cert_regions)

array(['D', 'E', 'F', 'G'], dtype=object)

array(['3 Certified', '2 Partial', '2 Substantial', '1 Limited',
       'Accredited', 'Failure of Activity', '0 Failed', '1 Failed',
       'For Termination'], dtype=object)

array(['Region I', 'Region II', 'Region III', 'Region IV-A', 'Region V',
       'Region VI', 'Region VII', 'Region IX', 'Region X', 'Region XI',
       'Region XII', 'Region XIII', 'NCR', 'CAR', 'BARMM', 'Region VIII',
       'MIMAROPA', 'NIR'], dtype=object)

In [8]:
# Manually counted the rows in the PDF. The table rows in the PDF matched
# the Claude-converted .CSV file
cert_d = cert[cert['annex'] == "D"]
print(cert_d.shape)

cert_e = cert[cert['annex'] == "E"]
print(cert_e.shape)

(84, 5)
(90, 5)


# 2.0. Process

In [11]:
display(cert.head(1))

,annex,esc_school_id,region,school_name,rating
0,D,104519,Region I,Academia De Sta. Cecilia Foundation Inc.,3 Certified


## 2.1. Clean

In [35]:
tmp_cert = cert.copy()

# Let's include the title of the annex
annex_title_map = {
    "D":"JHS Visited for Certification",
    "E":"JHS Visited for Certification Revisit",
    "F":"ESC Participating JHS Visited for Recertification",
    "G":"ESC Participating JHS Visited for Recertification Revisit"
}
tmp_cert['annex_label'] = tmp_cert['annex'].map(annex_title_map)

# Let's create new columns for a "rating" that's split
rating_map = {
    "Accredited": 1,
    "3 Certified": 2,
    "2 Substantial": 3,
    "2 Partial": 4,
    "1 Limited": 5,
    "1 Failed": 6,
    "0 Failed": 6,
    "Failure of Activity": 7,
    "For Termination": 8,
}
tmp_cert['rating_rank'] = tmp_cert['rating'].map(rating_map)

# Establish data types
str_cols = tmp_cert.columns[:-1]
int_cols = ['rating_rank']

for col in str_cols:
    tmp_cert[col] = tmp_cert[col].astype(str)
for col in int_cols:
    tmp_cert[col] = tmp_cert[col].astype(int)

print(tmp_cert.shape)
display(tmp_cert.head(3))

(899, 7)


,annex,esc_school_id,region,school_name,rating,annex_label,rating_rank
0,D,104519,Region I,Academia De Sta. Cecilia Foundation Inc.,3 Certified,JHS Visited for Certification,2
1,D,104558,Region I,Bugallon Kidsworld Academy Inc.,2 Partial,JHS Visited for Certification,4
2,D,104552,Region I,Ladder of Success Montessori School of Mangata...,2 Partial,JHS Visited for Certification,4


In [36]:
# check duplicate esc_school_id, if any
display(tmp_cert[tmp_cert['esc_school_id'].duplicated()])

,annex,esc_school_id,region,school_name,rating,annex_label,rating_rank


We follow the recommended hierarchical ranking of the ratings below for this cohort of ESC participatings schools.

From highest to lowest:
1. Accredited (FAAP accreditation is above PEAC certification)
2. 3 Certified (Full Compliance)
3. 2 Substantial (Substantial Compliance)
4. 2 Partial (Partial Compliance)
5. 1 Limited (Limited Compliance)
6. 1 Failed / 0 Failed (Non-compliance)
7. Failure of Activity (Did not complete certification)
8. For Termination (Being removed from program)

## 2.2. DepEd IDs

In [19]:
# Load GASTPE tuition dataset that has a mapping of ESC school ID with DepEd ID
fname = "ESC and SHSVP Tuition.xlsx"
fpath = str(PRIVATE_DIR / fname)
fees = pd.read_excel(fpath, sheet_name="Tuition", engine="calamine")
print(fees.shape)

(5188, 15)


In [22]:
display(fees.head(2))
# display(fees.tail(2))

,Region SHS,Program,SUC/LUC,Billed in SY 2024-2025 (ESC),ESC School ID,DepEd School Id,School Name,ESC (Tuition),ESC (Other),ESC (Misc),ESC (Total),SHSVP (Tuition),SHSVP (Other),SHSVP (Misc),SHSVP (Total)
0,Region I,BOTH,NaN,NaN,100008.0,400001,"St. Andrew Academy of Bacarra, Inc.",13353.05,3817.85,2220.0,19390.90,15515.0,3210.0,2220.0,20945.0
1,Region I,BOTH,NaN,NaN,100018.0,400002,"Badoc Junior College, Inc.",9808.26,908.98,0.0,10717.24,15000.0,2658.0,0.0,17658.0


In [37]:
# Process fees dataframe
relevant_cols = ['ESC School ID', 'DepEd School Id', 'School Name']
tmp_fees = fees[relevant_cols].copy()

# Rename column headers
tmp_fees.rename(
    columns={
        "ESC School ID":"esc_school_id",
        "DepEd School Id":"school_id",
        "School Name":"school_name",
    },
    inplace=True
)

# Remove decimal places from ESC School ID
tmp_fees['esc_school_id'] = tmp_fees['esc_school_id'].astype(str)
pattern = r"(.*)\.0"
tmp_fees['esc_school_id'] = tmp_fees['esc_school_id'].str.extract(pattern)

for col in tmp_fees.columns:
    tmp_fees[col] = tmp_fees[col].astype(str)

print(tmp_fees.shape)
display(tmp_fees.head(3))

(5188, 3)


,esc_school_id,school_id,school_name
0,100008,400001,"St. Andrew Academy of Bacarra, Inc."
1,100018,400002,"Badoc Junior College, Inc."
2,100023,400003,"IGAMA Colleges Foundation, Inc."


## 2.3 Merge DepEd IDs

In [38]:
# Certification is voluntary hence it is only a subset of all ESC delivering schools
cert_mrg = tmp_cert.merge(
    tmp_fees[['esc_school_id','school_id']],
    on='esc_school_id',
    how='left'
)
print(cert_mrg.shape)

(899, 8)


In [39]:
display(cert_mrg.head())

,annex,esc_school_id,region,school_name,rating,annex_label,rating_rank,school_id
0,D,104519,Region I,Academia De Sta. Cecilia Foundation Inc.,3 Certified,JHS Visited for Certification,2,NaN
1,D,104558,Region I,Bugallon Kidsworld Academy Inc.,2 Partial,JHS Visited for Certification,4,NaN
2,D,104552,Region I,Ladder of Success Montessori School of Mangata...,2 Partial,JHS Visited for Certification,4,NaN
3,D,104525,Region I,Mother of Good Counsel Academy Inc.,3 Certified,JHS Visited for Certification,2,NaN
4,D,104515,Region I,Precious Minds Montessori and High School Inc.,2 Substantial,JHS Visited for Certification,3,NaN


In [40]:
cert_mrg.isna().sum()

annex              0
esc_school_id      0
region             0
school_name        0
rating             0
annex_label        0
rating_rank        0
school_id        108
dtype: int64

In [53]:
cert_mrg.dtypes

annex            object
esc_school_id    object
region           object
school_name      object
rating           object
annex_label      object
rating_rank       int64
school_id        object
dtype: object

## 2.3. DepEd IDs pt2
We'll use our ESC Beneficiaries dataset that also contains a mapping of esc_school_id to DepEd school IDs.

In [43]:
fname = "processed_esc_beneficiaries.parquet"
fpath = str(OUTPUT_DIR / fname)
benef = pd.read_parquet(fpath, engine='fastparquet')
print(benef.shape)

(2681668, 14)


In [44]:
display(benef.head(1))

,old_region,destination_school_id,lrn,full_name,esc_school_id,school_name,grade_level,billing_statement_no,esc_subsidy_amount,source_sheet_name,lrn_category,origin_school_id,is_valid_origin_school_id,school_year
0,BARMM,475511,133526130177,Mae Ann Canoy Abajo,1603520,"Adiong Memorial College Foundation, Inc.",grade_9,ESC-227153,9000,BARMM,Standard,133526,True,-1 days +23:59:59.979777977


In [54]:
benef_ids = benef[['esc_school_id', 'destination_school_id']].copy()

# Drop duplicating esc_school_id to make a table with unique esc schools
benef_ids = benef_ids.drop_duplicates(subset='esc_school_id')

for col in benef_ids.columns:
    benef_ids[col] = benef_ids[col].astype(str)

print(benef_ids.shape)

(3767, 2)


In [55]:
crt_mrg = cert_mrg.copy()

crt_mrg = cert_mrg.merge(
    benef_ids,
    on='esc_school_id',
    how='left'
)
print(crt_mrg.shape)

(899, 9)


In [57]:
crt_mrg.isna().sum()

annex                      0
esc_school_id              0
region                     0
school_name                0
rating                     0
annex_label                0
rating_rank                0
school_id                108
destination_school_id     98
dtype: int64

## 2.4. Reconcile IDs
We were only able to reduce the 108 schools in the Certifications dataset that does not have a corresponding DepEd ID by 13 after using two GASS-PEAC datasets.

In [74]:
tmp_mrg = crt_mrg.copy()

mask = (tmp_mrg['school_id'].isna()) & (~tmp_mrg['destination_school_id'].isna())
mrg_na = tmp_mrg.loc[mask]

tmp_mrg.loc[mask, 'school_id'] = tmp_mrg.loc[mask, 'destination_school_id'].values

# Drop destination_school_id now
tmp_mrg.drop(columns=['destination_school_id'], inplace=True)

# Create a new column for validation of esc-deped ID pairs
tmp_mrg['has_deped_school_id'] = True

mask = tmp_mrg['school_id'].isna()
tmp_mrg.loc[mask, 'has_deped_school_id'] = False

# Reorder columns
ord_cols = ['school_id','esc_school_id','annex','annex_label','has_deped_school_id','rating_rank']
tmp_mrg = tmp_mrg[ord_cols]

print(mrg_na.shape)

(13, 9)


In [76]:
# display(tmp_mrg[mask])
display(tmp_mrg)

,school_id,esc_school_id,annex,annex_label,has_deped_school_id,rating_rank
0,NaN,104519,D,JHS Visited for Certification,False,2
1,NaN,104558,D,JHS Visited for Certification,False,4
2,NaN,104552,D,JHS Visited for Certification,False,4
3,NaN,104525,D,JHS Visited for Certification,False,2
4,NaN,104515,D,JHS Visited for Certification,False,3
...,...,...,...,...,...,...
894,448010,704038,G,ESC Participating JHS Visited for Recertificat...,True,5
895,404228,601063,G,ESC Participating JHS Visited for Recertificat...,True,2
896,404137,601077,G,ESC Participating JHS Visited for Recertificat...,True,2
897,404230,601064,G,ESC Participating JHS Visited for Recertificat...,True,2


In [77]:
tmp_mrg.isna().sum()

school_id              95
esc_school_id           0
annex                   0
annex_label             0
has_deped_school_id     0
rating_rank             0
dtype: int64

In [78]:
tmp_mrg['has_deped_school_id'].value_counts()

has_deped_school_id
True     804
False     95
Name: count, dtype: int64

In [82]:
tmp_mrg['rating_rank'].value_counts().sort_index()

rating_rank
1      2
2    506
3    250
4     69
5     48
6     19
7      2
8      3
Name: count, dtype: int64

## 2.5. Save

In [83]:
fpath = str(OUTPUT_DIR / "processed_esc_certification_rating.parquet")
tmp_mrg.to_parquet(fpath)
print(f"Successfully saved your file at {fpath}!")

Successfully saved your file at /workspace/project_paaral/output/processed_esc_certification_rating.parquet!
